# BİM448 Veri Madenciliği ve Bilgi Keşfi - Proje Ödevi
## 3. Yöntem: LLM Tabanlı Duygu Analizi (EVREN API - `qwen3.8-flash-next`)
- **9.000 İngilizce & 9.000 Türkçe Restoran/Kafe Yorumu**
- **Yöntem:** Yapılandırılmış İstem Mühendisliği (Structured Batch Prompting) & EVREN OpenAI-Uyumlu API (`https://evren-llmapi.ssyz.org.tr/v1`)

In [1]:
import os
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

LABELS = ["Positive", "Neutral", "Negative"]

def evaluate_llm(df, pred_col, title):
    y_true = df["true_label"]
    y_pred = df[pred_col]
    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    print("=" * 65)
    print(f" {title}")
    print("=" * 65)
    print(f"Genel Doğruluk (Accuracy)   : %{acc * 100:.2f} ({(y_true == y_pred).sum():,} / {len(df):,} doğru)")
    print(f"Toplam Hatalı Tahmin Sayısı : {(y_true != y_pred).sum():,} / {len(df):,}\n")
    for i, lbl in enumerate(LABELS):
        tot = int(cm[i].sum())
        cor = int(cm[i, i])
        err = tot - cor
        mis = ", ".join([f"{LABELS[j]}'e kayan: {int(cm[i, j])}" for j in range(len(LABELS)) if j != i])
        print(f"  - [{lbl:8s}] Toplam: {tot} | Doğru: {cor:4d} | Hatalı (Yanlış): {err:4d} (%{err/tot*100:5.2f} hata) -> ({mis})")
    print("\nKarmaşıklık Matrisi (Confusion Matrix):")
    print(pd.DataFrame(cm, index=[f"Gerçek {l}" for l in LABELS], columns=[f"Tahmin {l}" for l in LABELS]))
    print("\n" + classification_report(y_true, y_pred, labels=LABELS, digits=4))

In [2]:
# EVREN API ile üretilen İngilizce ve Türkçe LLM sonuçlarının değerlendirilmesi
df_en_llm = pd.read_csv("llm_english_results.csv")
df_tr_llm = pd.read_csv("llm_turkish_results.csv")

evaluate_llm(df_en_llm, "llm_pred", "İNGİLİZCE VERİ SETİ - LLM SONUÇLARI (9.000 Yorum)")
print()
evaluate_llm(df_tr_llm, "llm_pred", "TÜRKÇE VERİ SETİ - LLM SONUÇLARI (9.000 Yorum)")

Önbellekteki tamamlanmış sonuç dosyası kullanılıyor: llm_english_results.csv

 İNGİLİZCE VERİ SETİ - LLM TABANLI DUYGU ANALİZİ SONUÇLARI (9.000 Yorum)
 Model: qwen3.8-flash-next (EVREN API)
Genel Doğruluk (Accuracy)   : %83.63 (7,527 / 9,000 doğru)
Toplam Hatalı Tahmin Sayısı : 1,473 / 9,000

Sınıf Bazlı Hata Analizi (Gerçek Sınıfa Göre):
  - [Positive] Toplam: 3000 | Doğru: 2848 | Hatalı:  152 (% 5.07 hata) -> (Neutral'e kayan: 147, Negative'e kayan: 5)
  - [Neutral ] Toplam: 3000 | Doğru: 2074 | Hatalı:  926 (%30.87 hata) -> (Positive'e kayan: 495, Negative'e kayan: 431)
  - [Negative] Toplam: 3000 | Doğru: 2605 | Hatalı:  395 (%13.17 hata) -> (Positive'e kayan: 5, Neutral'e kayan: 390)

Karmaşıklık Matrisi (Confusion Matrix):
                 Tahmin Positive  Tahmin Neutral  Tahmin Negative
Gerçek Positive             2848             147                5
Gerçek Neutral               495            2074              431
Gerçek Negative                5             390             26